# 03 — PyTorch Autograd and Gradient Mechanics

**Session:** 2026-11-06 · Design Validation  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## Overview

Modern deep learning frameworks automate the computation of partial derivatives through **Reverse-Mode Automatic Differentiation** (`autograd`).
Understanding the computational Directed Acyclic Graph (DAG), tensor memory representations, forward graph accumulation, and backward gradient propagation is essential for debugging optimization bottlenecks and crafting custom loss functions.

This notebook explores:
1. **The Reverse-Mode Computational Graph:** Building a pedagogical autograd scalar engine from scratch.
2. **PyTorch Tensor Primitives:** Shapes, memory contiguous layouts, and operations.
3. **Linear Regression from Scratch:** Analytical gradient derivations vs. automated graph differentiation.
4. **Logistic Regression from Scratch:** Binary Cross-Entropy loss with numerical stabilizer epsilons.
5. **Modular Architecture:** Transitioning from raw tensors to `torch.nn.Module` and optimizers.

In [ ]:
from __future__ import annotations

import math
from collections.abc import Sequence

import numpy as np

np.random.seed(42)
print("Environment initialized with deterministic PRNG seed.")

## 1. Demystifying Autograd: Educational Micro-Autograd Engine

To understand how PyTorch's `backward()` propagates gradients, we implement a self-contained automatic differentiation scalar node: `Value`.
Every operation constructs an edge in a dynamic directed graph, recording the local partial derivative with respect to its inputs via the chain rule:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial x}$$

In [ ]:
class Value:
    """Educational scalar autograd node storing values, gradients, and backward closures."""

    def __init__(self, data: float, _children: Sequence[Value] = (), _op: str = ""):
        self.data: float = float(data)
        self.grad: float = 0.0
        self._backward = lambda: None
        self._prev: set[Value] = set(_children)
        self._op: str = _op

    def __repr__(self) -> str:
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other: Value | float) -> Value:
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other: Value | float) -> Value:
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, power: int | float) -> Value:
        out = Value(self.data**power, (self,), f"**{power}")

        def _backward():
            self.grad += (power * (self.data ** (power - 1))) * out.grad

        out._backward = _backward
        return out

    def relu(self) -> Value:
        out = Value(max(0.0, self.data), (self,), "ReLU")

        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad

        out._backward = _backward
        return out

    def backward(self) -> None:
        """Topological sort traversal executing reverse-mode chain rule accumulation."""
        topo: list[Value] = []
        visited: set[Value] = set()

        def build_topo(v: Value):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


# Test computational graph backward pass: L = (x * w + b)^2
x = Value(2.0)
w = Value(3.0)
b = Value(1.0)
z = x * w + b  # 2*3 + 1 = 7
loss = z**2  # 7^2 = 49
loss.backward()

print(f"Loss = {loss.data:.1f}")
print(f"dL/dw = 2 * z * x = 2 * 7 * 2 = 28.0 | Computed grad: {w.grad:.1f}")
print(f"dL/db = 2 * z * 1 = 2 * 7 * 1 = 14.0 | Computed grad: {b.grad:.1f}")
assert math.isclose(w.grad, 28.0) and math.isclose(b.grad, 14.0)

## 2. Vectorized Tensor Formulation for Supervised Optimization

Moving from scalar nodes to vectorized tensors, we express Linear Regression with Mean Squared Error (MSE):

$$\hat{Y} = X W + b$$

$$J(W, b) = \frac{1}{2N} \| \hat{Y} - Y \|_2^2 = \frac{1}{2N} (X W + b - Y)^T (X W + b - Y)$$

Analytical partial gradients derived via matrix calculus:

$$\nabla_W J = \frac{1}{N} X^T (\hat{Y} - Y), \quad \nabla_b J = \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i)$$

In [ ]:
class VectorizedLinearRegression:
    """Pure NumPy vectorized implementation matching PyTorch tensor mechanics."""

    def __init__(self, in_features: int, lr: float = 0.05):
        self.w = np.random.normal(0.0, 0.01, size=(in_features, 1))
        self.b = 0.0
        self.lr = lr

    def forward(self, x: np.ndarray) -> np.ndarray:
        return np.dot(x, self.w) + self.b

    def compute_loss(self, y_pred: np.ndarray, y_true: np.ndarray) -> float:
        return float(np.mean((y_pred - y_true) ** 2))

    def update_step(self, x: np.ndarray, y_pred: np.ndarray, y_true: np.ndarray):
        n = len(x)
        error = y_pred - y_true
        grad_w = np.dot(x.T, error) / n
        grad_b = float(np.mean(error))
        self.w -= self.lr * grad_w
        self.b -= self.lr * grad_b


# Synthesize linear regression dataset: y = 3.5 * x1 - 2.0 * x2 + 4.2 + noise
n_samples = 500
X_synth = np.random.uniform(-2.0, 2.0, size=(n_samples, 2))
true_w = np.array([[3.5], [-2.0]])
true_b = 4.2
y_synth = np.dot(X_synth, true_w) + true_b + np.random.normal(0, 0.1, size=(n_samples, 1))

model_lr = VectorizedLinearRegression(in_features=2, lr=0.1)
loss_history = []
for _epoch in range(150):
    preds = model_lr.forward(X_synth)
    loss = model_lr.compute_loss(preds, y_synth)
    loss_history.append(loss)
    model_lr.update_step(X_synth, preds, y_synth)

print(f"Learned weights: {model_lr.w.ravel().round(3)} | True weights: {true_w.ravel()}")
print(f"Learned bias: {model_lr.b:.3f} | True bias: {true_b}")
print(f"Final training MSE: {loss_history[-1]:.4f}")

## 3. Logistic Regression with Binary Cross-Entropy Loss

For binary classification $y \in \{0, 1\}$, we apply the logistic sigmoid function $\sigma(z)$:

$$\hat{p} = \sigma(X W + b) = \frac{1}{1 + e^{-(X W + b)}}$$

Binary Cross-Entropy Loss with numerical stabilization ($\\epsilon = 10^{-12}$):

$$J(W, b) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \ln(\hat{p}_i + \epsilon) + (1 - y_i) \ln(1 - \hat{p}_i + \epsilon) \right]$$

Analytical gradient:

$$\nabla_W J = \frac{1}{N} X^T (\hat{P} - Y)$$

In [ ]:
class VectorizedLogisticRegression:
    """Vectorized binary classification with numerically bounded Cross-Entropy."""

    def __init__(self, in_features: int, lr: float = 0.1):
        self.w = np.zeros((in_features, 1))
        self.b = 0.0
        self.lr = lr

    @staticmethod
    def sigmoid(z: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(z, -20.0, 20.0)))

    def forward(self, x: np.ndarray) -> np.ndarray:
        return self.sigmoid(np.dot(x, self.w) + self.b)

    def compute_loss(self, y_pred: np.ndarray, y_true: np.ndarray, eps: float = 1e-12) -> float:
        p = np.clip(y_pred, eps, 1.0 - eps)
        bce = -np.mean(y_true * np.log(p) + (1.0 - y_true) * np.log(1.0 - p))
        return float(bce)

    def update_step(self, x: np.ndarray, y_pred: np.ndarray, y_true: np.ndarray):
        n = len(x)
        error = y_pred - y_true
        grad_w = np.dot(x.T, error) / n
        grad_b = float(np.mean(error))
        self.w -= self.lr * grad_w
        self.b -= self.lr * grad_b

    def predict(self, x: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.forward(x) >= threshold).astype(int)


# Synthesize separable 2D classification dataset
np.random.seed(42)
n_pts = 300
class_0 = np.random.normal(loc=[-1.0, -1.0], scale=0.8, size=(n_pts, 2))
class_1 = np.random.normal(loc=[1.5, 1.5], scale=0.8, size=(n_pts, 2))
X_clf = np.vstack([class_0, class_1])
y_clf = np.vstack([np.zeros((n_pts, 1)), np.ones((n_pts, 1))])

log_model = VectorizedLogisticRegression(in_features=2, lr=0.2)
for _epoch in range(200):
    probs = log_model.forward(X_clf)
    loss = log_model.compute_loss(probs, y_clf)
    log_model.update_step(X_clf, probs, y_clf)

acc = float(np.mean(log_model.predict(X_clf) == y_clf))
print(f"Classification accuracy: {acc * 100:.2f}% | Final Loss: {loss:.4f}")

## 4. PyTorch Computational API Representation

In deep learning workflows using PyTorch, the operations above are structured with `nn.Module`, `optim.Optimizer`, and `LossFunction` abstractions:

```python
import torch
import torch.nn as nn
import torch.optim as optim


class LogisticRegressionModule(nn.Module):

    def __init__(self, in_features: int):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.linear(x))


model = LogisticRegressionModule(in_features=2)
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training step:
optimizer.zero_grad()  # Reset accumulated gradients
outputs = model(x_tensor)  # Forward pass through DAG
loss = criterion(outputs, y_tensor)  # Compute scalar loss
loss.backward()  # Reverse-mode chain rule propagation
optimizer.step()  # Parameter update step
```

### Key Rules of PyTorch Gradient Hygiene:
1. Always call `optimizer.zero_grad()` before `loss.backward()`, as PyTorch accumulates gradients in `.grad` by default.
2. In evaluation loops, wrap execution in `with torch.no_grad():` to disable graph building and reduce memory overhead by ~60%.
3. Explicitly toggle `model.train()` and `model.eval()` to control stateful layers like Dropout and BatchNorm.